In [1]:
import duckdb
import pandas as pd

In [2]:
#   Analysing a particular scenario
OBSERVATIONS_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/observations.jsonl"
LABELS_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/labels.jsonl"
SUPPORTING_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/supporting.jsonl"

#   JSON
SNAPSHOT_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/snapshot.json"

In [3]:
def read_scenario(observations_path: str, labels_path: str, supporting_path: str, snapshot_path: str):
    observations_df = pd.read_json(observations_path, lines=True)
    attributes_df = pd.json_normalize(observations_df["attributes"]).add_prefix("observ_")
    observations_df = pd.concat(
        [observations_df.drop(columns=["attributes"]), attributes_df],
        axis=1
    )

    labels_df = pd.read_json(labels_path, lines=True)

    supporting_df = pd.read_json(supporting_path, lines=True)
    attributes_df_2 = pd.json_normalize(supporting_df["attributes"]).add_prefix("supporting_")
    supporting_df = pd.concat(
        [supporting_df.drop(columns=["attributes"]), attributes_df_2],
        axis=1
    )
    
    snapshot_df = pd.read_json(snapshot_path)       

    return observations_df, labels_df, supporting_df, snapshot_df


In [4]:
def process_sql_query(query: str):
    return duckdb.query(query).df()

In [5]:
observations_df, labels_df, supporting_df, snapshot_df = read_scenario(
    OBSERVATIONS_PATH,
    LABELS_PATH,
    SUPPORTING_PATH,
    SNAPSHOT_PATH
)

print(observations_df.shape)
print(labels_df.shape)
print(supporting_df.shape)
print(snapshot_df.shape)

(322, 18)
(24, 2)
(192, 8)
(3, 10)


#### Analysing the observations dataframe

In [6]:
observations_df.head(3)

,observation_id,tracklet_id,observed_at,received_at,latitude,longitude,source,altitude_ft,observ_c2,observ_ref,observ_q,observ_node,observ_r7,observ_v,observ_hdg,observ_tag,observ_spd,observ_crs
0,obs-2e9f4ac3-91f7-9cff-0446-0ab004882891,tracklet-27f73aa2-d32d-3652-9edb-b41e281835c3,2026-01-10 00:12:04.135+00:00,2026-01-10 00:12:15.050+00:00,35.020750,-108.316764,feed-b,18683.2,230.74,2817A5,1,worker-3,-0.431,NaN,NaN,NaN,NaN,NaN
1,obs-80a29fa0-b5f3-f921-4eac-3a8eeff78697,tracklet-661a285a-0cde-2cff-c9ba-fcec04a77d92,2026-01-10 00:05:53.641+00:00,2026-01-10 00:06:07.592+00:00,35.389153,-107.808179,feed-c,NaN,NaN,NaN,NaN,worker-14,4.301,589.24,235.93,D32EA2,NaN,NaN
2,obs-4dad52d0-3c9c-28be-39bd-047c213314b7,tracklet-aefc9cdb-3772-235d-2f29-89f456dd9ca9,2026-01-10 00:01:49.698+00:00,2026-01-10 00:01:55.514+00:00,35.587849,-107.451235,feed-a,16471.8,NaN,D32EA2,2,worker-9,-0.169,NaN,NaN,NaN,306.57,234.76


In [7]:
observations_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   observation_id  322 non-null    str                
 1   tracklet_id     322 non-null    str                
 2   observed_at     322 non-null    datetime64[us, UTC]
 3   received_at     322 non-null    datetime64[us, UTC]
 4   latitude        322 non-null    float64            
 5   longitude       322 non-null    float64            
 6   source          322 non-null    str                
 7   altitude_ft     251 non-null    float64            
 8   observ_c2       80 non-null     float64            
 9   observ_ref      211 non-null    str                
 10  observ_q        295 non-null    object             
 11  observ_node     322 non-null    str                
 12  observ_r7       322 non-null    float64            
 13  observ_v        84 non-null     float64       

In [8]:
observations_df.isna().sum()

observation_id      0
tracklet_id         0
observed_at         0
received_at         0
latitude            0
longitude           0
source              0
altitude_ft        71
observ_c2         242
observ_ref        111
observ_q           27
observ_node         0
observ_r7           0
observ_v          238
observ_hdg        235
observ_tag        235
observ_spd        119
observ_crs        196
dtype: int64

In [9]:
#   Although there are 322 total observations, only 312 of them are unique!!

observations_df["observation_id"].nunique()     #   This info is also present in snapshot.json

312

In [10]:
#   Although there are 322 total rows in the df, only 24 tracklets of them are unique!!
#   So, there are 24 tracklets in total

observations_df["tracklet_id"].nunique()        #   This info is also present in snapshot.json

24

### Analysing the supporting dataframe

In [11]:
supporting_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   record_id       192 non-null    str                
 1   observation_id  192 non-null    str                
 2   source          192 non-null    str                
 3   received_at     192 non-null    datetime64[us, UTC]
 4   supporting_a1   166 non-null    float64            
 5   supporting_spd  192 non-null    int64              
 6   supporting_b4   192 non-null    str                
 7   supporting_rx   192 non-null    float64            
dtypes: datetime64[us, UTC](1), float64(2), int64(1), str(4)
memory usage: 12.1 KB


In [12]:
supporting_df.isna().sum()

record_id          0
observation_id     0
source             0
received_at        0
supporting_a1     26
supporting_spd     0
supporting_b4      0
supporting_rx      0
dtype: int64

In [13]:
supporting_df.head()

,record_id,observation_id,source,received_at,supporting_a1,supporting_spd,supporting_b4,supporting_rx
0,support-109ae044-1a9c-2f0f-c675-ab3650fbf337,obs-af12d592-a209-265a-8c06-6d14441d978c,telemetry-x,2026-01-10 00:06:19.480+00:00,4222.7,74,ok,-67.5
1,support-b4855af0-6d5a-fcfe-12c8-abe607a2a3ef,obs-d2c1d9a2-c2dd-53ff-5633-1593c011c4e5,telemetry-x,2026-01-10 00:04:11.773+00:00,5762.6,3,ok,-55.4
2,support-62b22063-f999-a608-3152-5d1466ee5e67,obs-d85393bc-4e11-298e-2deb-a98257f053da,telemetry-x,2026-01-10 00:08:09.729+00:00,4210.5,70,hold,-69.1
3,support-9662c794-130f-8591-bb7b-13b76bf3e152,obs-80a29fa0-b5f3-f921-4eac-3a8eeff78697,telemetry-x,2026-01-10 00:06:08.005+00:00,4808.3,84,hold,-68.7
4,support-013e9d4d-3ad9-ffaa-94d0-e7cba5e62653,obs-31f836ae-f00f-7b52-270b-8298f825356b,telemetry-x,2026-01-10 00:04:19.174+00:00,3699.3,93,ok,-67.9


In [14]:
supporting_df.shape     #   This info is already given in snapshot.jsonl

(192, 8)

In [15]:
#   Out of 192 rows, there are only 186 records
supporting_df["record_id"].nunique()    #   This info IS NOT PRESENT in snapshot.json

186

In [16]:
#   For each record, there is an observation
supporting_df["observation_id"].nunique()   

186

In [17]:
#   Checking how many unique pairs for observation and records are there
supporting_df[["record_id", "observation_id"]].drop_duplicates().shape[0]

#   This result concludes that for every record, there is an observation!!

186

In [18]:
#   Checking the unique sources
supporting_df["source"].unique().tolist()

['telemetry-x']

### Investigating labels_df

In [19]:
labels_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   tracklet_id            24 non-null     str  
 1   successor_tracklet_id  18 non-null     str  
dtypes: str(2)
memory usage: 516.0 bytes


In [20]:
labels_df.isna().sum()

tracklet_id              0
successor_tracklet_id    6
dtype: int64

In [21]:
labels_df.head()

,tracklet_id,successor_tracklet_id
0,tracklet-03b4b60d-f1d1-be22-62ea-070dd9b06819,NaN
1,tracklet-054df159-4c90-c17e-91a4-1ed8c483c568,tracklet-f1642a8a-d627-2e56-04ff-48943b13adba
2,tracklet-0d60a69d-4493-259d-deb2-9d191a81dde6,tracklet-3d790901-6bbf-dc78-bf5b-417056a62b02
3,tracklet-25a03cd1-b106-0bba-6635-f770f85253dd,tracklet-e293fd97-9453-28c8-9275-e5ba64125b1b
4,tracklet-27f73aa2-d32d-3652-9edb-b41e281835c3,NaN


In [22]:
labels_df["tracklet_id"].nunique()

24

In [23]:
labels_df["successor_tracklet_id"].nunique()

18

In [24]:
#   Checking how many successor_tracklet_ids are present in the original 24 tracklet_ids

process_sql_query(
"""
    WITH A AS (
        SELECT DISTINCT tracklet_id AS id
        FROM labels_df
    ),
    B AS (
        SELECT DISTINCT successor_tracklet_id AS id
        FROM labels_df
        WHERE successor_tracklet_id IS NOT NULL
    )
    SELECT
        (SELECT COUNT(*) FROM A) AS count_A,
        (SELECT COUNT(*) FROM B) AS count_B,
        (SELECT COUNT(*) FROM A INNER JOIN B USING (id)) AS intersection_count
"""
)

#   This result confirms that successor_tracklet_id is just a subset of the tracklet_ids (which makes sense as well..!!)

,count_A,count_B,intersection_count
0,24,18,18


#### Observations:
1. For observations_df, the de-duplicated info is already given in snapshot.jsonl
2. But for supporting_df, it is not given


#### Probable next steps:
1. Add an extra field in snapshot.jsonl => the missing info from supporting_df => the unique no. of observations
2. Track and investigate the `successor_tracklet_id` in labels_df and how does that make sense
3. Also need to understand the meaning of the attributes in observation_df and supporting_df
4. Join the observations_df and supporting_df over `observation_id` (to bring supporting information)
5. Join the above df with labels_df over `tracklet_id` (to bring the labels)
6. Remember, we need to preserve all obsservations (cross-check??), so inner join may not be appropriate. Investigate this


In [25]:
#   Important Notes:

#   From observations_df:
#       1. unique_observation_count: 312
#       2. observation_row_count: 322
#       3. unique_tracklet_count: 24


#   From supporting_df:
#       1. unique_observation_count: 186

#   From labels_df:
#       1. unique_tracklet_count: 24

#   a. This indicates that some observations don't have supporting data
#   b. This also indicates that all observations do have their corresponding labels


#      Coordinate Validation: You must validate coordinate ranges. If a latitude is out of bounds (e.g., greater than 90 or less than -90), you must reject that row. There is actually a specific test file mentioned (fixtures/invalid_coordinate.jsonl) designed to check if your code catches this.
#      Missing Altitude: The document explicitly warns: "Do not silently treat missing values as zero." Altitude can be null. If you fill it with a 0, your model might think the aircraft suddenly dove to sea level. You need to handle it explicitly (e.g., using interpolation, maintaining it as a NaN, or dropping it if it's too sparse).
#      Strict Separation of Time: Ensure you are strictly using observed_at to calculate physical things like distance and speed. received_at is only when the server received the data—you will only need received_at later when you do the "staged demonstration" to simulate delayed data.
#      Drop IDs before ML: While you need tracklet_id, observation_id, and scenario_id to join and group the data, you must not use them as predictive features for your ML model.

In [26]:
process_sql_query(
    """
SELECT
    (SELECT COUNT(*) FROM observations_df) AS observation_rows,
    (SELECT COUNT(DISTINCT observation_id) FROM observations_df) AS unique_observations,
    (SELECT COUNT(DISTINCT tracklet_id) FROM observations_df) AS observation_tracklets,

    (SELECT COUNT(DISTINCT observation_id) FROM supporting_df) AS supported_observations,

    (SELECT COUNT(DISTINCT tracklet_id) FROM labels_df) AS labeled_tracklets;
"""
)

,observation_rows,unique_observations,observation_tracklets,supported_observations,labeled_tracklets
0,322,312,24,186,24


In [27]:
process_sql_query(
"""
    WITH
-- Unique observations
O AS (
    SELECT DISTINCT observation_id, tracklet_id
    FROM observations_df
),

-- Unique supporting observations
S AS (
    SELECT DISTINCT observation_id
    FROM supporting_df
),

-- Unique labeled tracklets
L AS (
    SELECT DISTINCT tracklet_id
    FROM labels_df
),

-- Unique tracklets in observations
OT AS (
    SELECT DISTINCT tracklet_id
    FROM observations_df
),

-- Supporting observations missing from observations_df
SO AS (
    SELECT DISTINCT s.observation_id
    FROM S s
    ANTI JOIN O o USING (observation_id)
),

-- Observations missing supporting data
OS AS (
    SELECT DISTINCT o.observation_id
    FROM O o
    ANTI JOIN S s USING (observation_id)
),

-- Observation tracklets missing labels
OL AS (
    SELECT DISTINCT o.tracklet_id
    FROM OT o
    ANTI JOIN L l USING (tracklet_id)
),

-- Labels with no corresponding observation tracklet
LO AS (
    SELECT DISTINCT l.tracklet_id
    FROM L l
    ANTI JOIN OT o USING (tracklet_id)
)

SELECT
    -- observations_df
    (SELECT COUNT(*) FROM observations_df) AS observation_row_count,
    (SELECT COUNT(*) FROM O) AS unique_observation_count,
    (SELECT COUNT(*) FROM OT) AS unique_observation_tracklet_count,

    -- supporting_df
    (SELECT COUNT(*) FROM S) AS unique_supporting_observation_count,

    -- labels_df
    (SELECT COUNT(*) FROM L) AS unique_labeled_tracklet_count,

    -- Observation ↔ Supporting verification
    (SELECT COUNT(*) FROM OS) AS observations_without_supporting,
    (SELECT COUNT(*) FROM SO) AS supporting_without_observation,
    (SELECT COUNT(*) FROM O ANTI JOIN S USING (observation_id))
        AS unsupported_observation_count,

    -- Observation ↔ Labels verification
    (SELECT COUNT(*) FROM OL) AS observation_tracklets_without_labels,
    (SELECT COUNT(*) FROM LO) AS labels_without_observation_tracklets,
    (SELECT COUNT(*) FROM OT INNER JOIN L USING (tracklet_id))
        AS matched_tracklet_count;
"""
)

,observation_row_count,unique_observation_count,unique_observation_tracklet_count,unique_supporting_observation_count,unique_labeled_tracklet_count,observations_without_supporting,supporting_without_observation,unsupported_observation_count,observation_tracklets_without_labels,labels_without_observation_tracklets,matched_tracklet_count
0,322,312,24,186,24,126,0,126,0,0,24


In [28]:
#   Checking the duplicate rows

process_sql_query(
"""
    SELECT
        observation_id,
        COUNT(*) AS occurrences
    FROM observations_df
    GROUP BY observation_id
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC;
"""
)

,observation_id,occurrences
0,obs-6d7ba3c9-b95e-570c-290f-6b893d262aa3,2
1,obs-4ca868eb-985e-a483-2fb7-0df947720ad9,2
2,obs-a34245b2-ee67-352e-ec2b-0c2fbfdbb347,2
3,obs-dd51c529-d693-45ab-3af3-b54dea99cdf6,2
4,obs-65b7ba51-fc5b-d47d-2ef5-07764bcf1a3e,2
5,obs-5df688d3-88d5-ea5c-a2df-fff8a68550ba,2
6,obs-282f91cc-840d-0b97-2897-6d243bcbd852,2
7,obs-d6d7d976-c1a8-483a-c4d5-00913fd8a77a,2
8,obs-9d60da47-8ee4-bcf2-26e6-bcda7dc42044,2
9,obs-c6f20254-5f95-6d5a-d159-a1ebdf771acf,2


In [29]:
process_sql_query(
    """
SELECT
    observation_id,
    COUNT(*) AS occurrences,
    COUNT(DISTINCT tracklet_id) AS distinct_tracklets
FROM observations_df
GROUP BY observation_id
HAVING COUNT(*) > 1;
"""
)

,observation_id,occurrences,distinct_tracklets
0,obs-a34245b2-ee67-352e-ec2b-0c2fbfdbb347,2,1
1,obs-282f91cc-840d-0b97-2897-6d243bcbd852,2,1
2,obs-65b7ba51-fc5b-d47d-2ef5-07764bcf1a3e,2,1
3,obs-5df688d3-88d5-ea5c-a2df-fff8a68550ba,2,1
4,obs-c6f20254-5f95-6d5a-d159-a1ebdf771acf,2,1
5,obs-6d7ba3c9-b95e-570c-290f-6b893d262aa3,2,1
6,obs-dd51c529-d693-45ab-3af3-b54dea99cdf6,2,1
7,obs-d6d7d976-c1a8-483a-c4d5-00913fd8a77a,2,1
8,obs-9d60da47-8ee4-bcf2-26e6-bcda7dc42044,2,1
9,obs-4ca868eb-985e-a483-2fb7-0df947720ad9,2,1


In [30]:
#   Checking if the duplicate records are complete duplicates
#   We need to perform de-duplication before proceeding further...!!

duplicate_observations = (
    observations_df[
        observations_df.duplicated("observation_id", keep=False)
    ]
    .sort_values("observation_id")
)

duplicate_observations

,observation_id,tracklet_id,observed_at,received_at,latitude,longitude,source,altitude_ft,observ_c2,observ_ref,observ_q,observ_node,observ_r7,observ_v,observ_hdg,observ_tag,observ_spd,observ_crs
284,obs-282f91cc-840d-0b97-2897-6d243bcbd852,tracklet-85f6360e-64cd-6f31-8cbd-b37b03fb30b6,2026-01-10 00:12:30.951+00:00,2026-01-10 00:12:35.578+00:00,35.230914,-108.688057,feed-c,15258.2,NaN,NaN,H,worker-3,1.787,670.17,245.47,DC8A69,NaN,NaN
286,obs-282f91cc-840d-0b97-2897-6d243bcbd852,tracklet-85f6360e-64cd-6f31-8cbd-b37b03fb30b6,2026-01-10 00:12:30.951+00:00,2026-01-10 00:12:35.578+00:00,35.230914,-108.688057,feed-c,15258.2,NaN,NaN,H,worker-3,1.787,670.17,245.47,DC8A69,NaN,NaN
37,obs-4ca868eb-985e-a483-2fb7-0df947720ad9,tracklet-d64c3cf9-bb19-6268-ca11-a3b5056fe010,2026-01-10 00:13:25.480+00:00,2026-01-10 00:13:27.110+00:00,35.148230,-108.303945,feed-b,14476.9,236.68,0B4CD4,0,worker-9,9.813,NaN,NaN,NaN,346.94,NaN
198,obs-4ca868eb-985e-a483-2fb7-0df947720ad9,tracklet-d64c3cf9-bb19-6268-ca11-a3b5056fe010,2026-01-10 00:13:25.480+00:00,2026-01-10 00:13:27.110+00:00,35.148230,-108.303945,feed-b,14476.9,236.68,0B4CD4,0,worker-9,9.813,NaN,NaN,NaN,346.94,NaN
219,obs-5df688d3-88d5-ea5c-a2df-fff8a68550ba,tracklet-d274d7bc-0d29-d032-fd56-edf5ce536b98,2026-01-10 00:08:28.155+00:00,2026-01-10 00:08:30.763+00:00,35.378828,-108.235078,feed-a,NaN,NaN,DC8A69,NaN,worker-3,-1.196,NaN,NaN,NaN,NaN,247.83
190,obs-5df688d3-88d5-ea5c-a2df-fff8a68550ba,tracklet-d274d7bc-0d29-d032-fd56-edf5ce536b98,2026-01-10 00:08:28.155+00:00,2026-01-10 00:08:30.763+00:00,35.378828,-108.235078,feed-a,NaN,NaN,DC8A69,NaN,worker-3,-1.196,NaN,NaN,NaN,NaN,247.83
131,obs-65b7ba51-fc5b-d47d-2ef5-07764bcf1a3e,tracklet-054df159-4c90-c17e-91a4-1ed8c483c568,2026-01-10 00:01:56.133+00:00,2026-01-10 00:02:03.541+00:00,35.615168,-107.501417,feed-a,15301.8,NaN,DC8A69,2,worker-9,-9.035,NaN,NaN,NaN,368.09,252.93
274,obs-65b7ba51-fc5b-d47d-2ef5-07764bcf1a3e,tracklet-054df159-4c90-c17e-91a4-1ed8c483c568,2026-01-10 00:01:56.133+00:00,2026-01-10 00:02:03.541+00:00,35.615168,-107.501417,feed-a,15301.8,NaN,DC8A69,2,worker-9,-9.035,NaN,NaN,NaN,368.09,252.93
221,obs-6d7ba3c9-b95e-570c-290f-6b893d262aa3,tracklet-f1642a8a-d627-2e56-04ff-48943b13adba,2026-01-10 00:03:57.872+00:00,2026-01-10 00:04:20.539+00:00,35.545535,-107.730773,feed-c,NaN,NaN,NaN,M,worker-3,3.212,651.85,251.45,DC8A69,NaN,NaN
109,obs-6d7ba3c9-b95e-570c-290f-6b893d262aa3,tracklet-f1642a8a-d627-2e56-04ff-48943b13adba,2026-01-10 00:03:57.872+00:00,2026-01-10 00:04:20.539+00:00,35.545535,-107.730773,feed-c,NaN,NaN,NaN,M,worker-3,3.212,651.85,251.45,DC8A69,NaN,NaN


In [31]:
supporting_df[
    supporting_df.duplicated("observation_id", keep=False)
].sort_values("observation_id")

,record_id,observation_id,source,received_at,supporting_a1,supporting_spd,supporting_b4,supporting_rx
35,support-73fd963e-9672-594e-e0b0-49a7e8521222,obs-1b660ce1-cbe5-cbb1-2d61-d8ab9773f653,telemetry-x,2026-01-10 00:02:11.319+00:00,3620.8,37,hold,-67.4
71,support-73fd963e-9672-594e-e0b0-49a7e8521222,obs-1b660ce1-cbe5-cbb1-2d61-d8ab9773f653,telemetry-x,2026-01-10 00:02:11.319+00:00,3620.8,37,hold,-67.4
38,support-809a3eaa-f7c4-5c0c-7e39-fd0477fecd94,obs-50dc24eb-3a8f-bbbc-d728-4b0071a8f822,telemetry-x,2026-01-10 00:05:55.912+00:00,4258.8,93,hold,-81.1
81,support-809a3eaa-f7c4-5c0c-7e39-fd0477fecd94,obs-50dc24eb-3a8f-bbbc-d728-4b0071a8f822,telemetry-x,2026-01-10 00:05:55.912+00:00,4258.8,93,hold,-81.1
3,support-9662c794-130f-8591-bb7b-13b76bf3e152,obs-80a29fa0-b5f3-f921-4eac-3a8eeff78697,telemetry-x,2026-01-10 00:06:08.005+00:00,4808.3,84,hold,-68.7
178,support-9662c794-130f-8591-bb7b-13b76bf3e152,obs-80a29fa0-b5f3-f921-4eac-3a8eeff78697,telemetry-x,2026-01-10 00:06:08.005+00:00,4808.3,84,hold,-68.7
14,support-9a5fc3ba-1cff-90ee-6576-0fbfe5f62d5a,obs-a8f0c90f-1fb5-1321-e460-878b7fea4d94,telemetry-x,2026-01-10 00:12:40.756+00:00,4000.6,9,ok,-54.6
119,support-9a5fc3ba-1cff-90ee-6576-0fbfe5f62d5a,obs-a8f0c90f-1fb5-1321-e460-878b7fea4d94,telemetry-x,2026-01-10 00:12:40.756+00:00,4000.6,9,ok,-54.6
34,support-9950dae5-f3ea-7530-3efe-e63bb6924854,obs-ee39d3ff-3a9c-df0b-28d0-9784967ec63c,telemetry-x,2026-01-10 00:04:56.000+00:00,3881.9,52,na,-79.5
58,support-9950dae5-f3ea-7530-3efe-e63bb6924854,obs-ee39d3ff-3a9c-df0b-28d0-9784967ec63c,telemetry-x,2026-01-10 00:04:56.000+00:00,3881.9,52,na,-79.5
